In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.7 Electromagnetic Induction

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.7",
    title="Electromagnetic Induction",
    blurb="A changing magnetic field makes an electric field that pushes charges "
    "around loops: Faraday's law, Lenz's law, and the moment the electric field "
    "stops being conservative — ∇×E = −∂B/∂t, the third Maxwell equation.",
    difficulty="advanced",
    estimate="110–140 min",
)

## Notebook overview

Until now the electric field has been **conservative**. Charges made it, it pointed
from them, and the work it did around any closed path was zero, which is exactly what
let us define a single-valued scalar potential $V$ in [§3.2](potential-energy.ipynb).
This notebook is where that
picture breaks, and it breaks on purpose: a **changing magnetic field makes an
electric field**, and that induced field does not emanate from charges, it
**circulates**. Its line integral around a loop is no longer zero. That one fact,
Faraday's law of induction, is the conceptual hinge of the volume.

We build it in stages. First the integral law: a changing magnetic flux through a loop
drives an electromotive force, $\mathcal{E}=-d\Phi_B/dt$, with the minus sign of
Lenz's law enforcing energy conservation. Then the same physics seen from a moving
conductor (motional EMF), and the payoff every power grid runs on, the AC generator.
Then the local form, **$\nabla\times\mathbf E=-\partial_t\mathbf B$**, obtained from
the integral law by Stokes' theorem. This is the third of Maxwell's four equations and
the first to couple $\mathbf E$ and $\mathbf B$ dynamically: where
[§3.2](potential-energy.ipynb) had
$\nabla\times\mathbf E=0$, we now find $\nabla\times\mathbf E\neq0$, so $\mathbf E$ is
no longer the gradient of a potential alone. We make that failure of path-independence
concrete and numerical, the very property [§3.2](potential-energy.ipynb) warned would
not survive induction.

Finally the practical consequences: **inductance** (a circuit's flux opposing changes
in its own current, and the mutual coupling between circuits), the magnetic field
energy $\tfrac12 LI^2$ that mirrors the electric field energy of
[§3.2](potential-energy.ipynb), and the
eddy-current braking that makes a magnet drift slowly down a copper pipe. We close by
collecting three of the four field equations, with the fourth, and light itself, one
notebook away.

Everything is in **SI units**, with $\mu_0=4\pi\times10^{-7}\,$T·m/A. Induction is the
place motion returns as genuine physics, so exactly one figure here is animated, the
changing flux of Exercise 1; everything else is a still.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: an induced EMF matching $-d\Phi/dt$, a Lenz sign opposing the
> change, a numerical curl equal to $-\partial_t\mathbf B$, a circulation that refuses
> to vanish, a mutual inductance matching the coaxial-loop formula. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*, not a verdict.
>
> **A numerical-differentiation note.** Several checks differentiate a sampled signal
> with `numpy.gradient`. Its one-sided stencils at the first and last sample are
> less accurate than the centred stencil used in the interior, so we validate on the
> **interior** (excluding the two end points), the same array-edge subtlety met when
> taking divergences and curls on a finite grid.

> **Scope.** A working review, not a full course. See Nolting, *Theoretical Physics 3*
> {cite}`nolting3`; Griffiths, *Introduction to Electrodynamics* {cite}`griffiths_em`
> (ch. 7); Jackson {cite}`jackson` (ch. 5–6).

## Theory in brief

### Faraday's law

The **magnetic flux** through a loop spanning a surface $S$ is $\Phi_B=\int_S
\mathbf B\cdot d\mathbf A$. Faraday's law says a *changing* flux drives an
electromotive force around the loop's boundary,

```{math}
:label: eq-faraday
\mathcal{E} = -\frac{d\Phi_B}{dt}.
```

The flux can change three ways: $\mathbf B$ itself changes in time, the loop's area
changes, or the loop moves or rotates in a non-uniform or angled field. All three
appear below.

### Lenz's law: the minus sign

The sign in {eq}`eq-faraday` is **Lenz's law**:

```{math}
:label: eq-lenz
\text{the induced current opposes the change in flux that produced it.}
```

If the flux is increasing, the induced current makes a field opposing the increase; if
decreasing, it props the flux up. This is energy conservation in disguise: a current
that *reinforced* the change would amplify itself without limit and deliver free
energy. The induced EMF therefore always carries the sign opposite to $d\Phi_B/dt$.

### Motional EMF

When a conductor moves through a field, the magnetic force $q\,\mathbf v\times\mathbf
B$ on its charge carriers pushes them along the conductor, acting as a battery:

```{math}
:label: eq-motional
\mathcal{E} = \oint (\mathbf v\times\mathbf B)\cdot d\boldsymbol\ell .
```

For a bar of length $L$ sliding at speed $v$ perpendicular to a uniform field $B$ this
is $\mathcal{E}=BLv$, and it agrees exactly with the flux rule {eq}`eq-faraday`
applied to the growing circuit area, two pictures of one phenomenon.

### The differential form: the hinge

Stokes' theorem turns $\oint\mathbf E\cdot d\boldsymbol\ell=-d\Phi_B/dt$ into a local
statement,

```{math}
:label: eq-curlE
\nabla\times\mathbf E = -\frac{\partial\mathbf B}{\partial t}.
```

**This is the hinge of the volume.** In electrostatics $\nabla\times\mathbf E=0$: the
field is conservative, a scalar potential exists, and $\oint\mathbf E\cdot
d\boldsymbol\ell=0$, the path-independence of [§3.2](potential-energy.ipynb). The
moment $\mathbf B$ varies in
time, $\nabla\times\mathbf E\neq0$: the line integral around a loop is no longer zero,
$\mathbf E$ is **no longer conservative**, and a single-valued scalar potential no
longer suffices. The induced field **circulates** in closed loops; it does not start
or end on charges. This is the third of Maxwell's four equations, and the first that
couples $\mathbf E$ and $\mathbf B$ dynamically.

### Inductance and magnetic energy

A circuit's own current makes a flux through itself; the proportionality is the
**self-inductance** $L$, and $\mathcal{E}=-L\,dI/dt$. Between two circuits the
**mutual inductance** $M$ relates the current in one to the flux it sends through the
other,

```{math}
:label: eq-inductance
\Phi_2 = M\,I_1, \qquad \mathcal{E}_2 = -M\,\frac{dI_1}{dt}.
```

Establishing a current against the induced back-EMF stores energy $\tfrac12 LI^2$ in
the field, with density $u=B^2/2\mu_0$, the magnetic mirror of the electric field
energy $(\varepsilon_0/2)|\mathbf E|^2$ of [§3.2](potential-energy.ipynb).

### Toward the unified potential (3.8)

Once $\nabla\times\mathbf E\neq0$, the field cannot come from $V$ alone. The fix uses
the vector potential of [§3.6](magnetostatics.ipynb) alongside the scalar one,
$\mathbf E=-\nabla V-\partial_t\mathbf A$, so that $V$ and $\mathbf A$ together carry
the field. That is the full gauge structure assembled in
[§3.8](maxwell-waves.ipynb); here we only plant the
pointer.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp

from ecp import draw, validate
from ecp.animate import show

from scipy.constants import mu_0 as MU0  # vacuum permeability, T·m/A

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT


def flux_uniform(B, area, angle=0.0):
    """Flux of a uniform field through a flat loop, Φ = B·A·cos θ.

    The magnetic flux whose rate of change drives the induced EMF (Faraday).

    Parameters
    ----------
    B : float or numpy.ndarray
        Field magnitude (may be time-dependent).
    area : float
        Loop area.
    angle : float or numpy.ndarray, optional
        Angle between the field and the loop normal (default 0).

    Returns
    -------
    float or numpy.ndarray
        The flux through the loop, in webers.
    """
    return B * area * np.cos(angle)


def biot_savart(path, current, P):
    """Biot-Savart field of a current path (callback to §3.6).

    Sums (μ0·I/4π)·dl × R̂/R^2 (`numpy.cross`) over the path segments; used
    here for the mutual-inductance flux.

    Parameters
    ----------
    path : numpy.ndarray
        Points ``(M, 3)`` tracing the wire.
    current : float
        Current in amperes.
    P : numpy.ndarray
        Field points of shape ``(..., 3)``.

    Returns
    -------
    numpy.ndarray
        The field at ``P``, in tesla.
    """
    path = np.asarray(path, float)
    P = np.asarray(P, float)
    mids = 0.5 * (path[1:] + path[:-1])
    dls = path[1:] - path[:-1]
    B = np.zeros_like(P)
    for dl, mid in zip(dls, mids):
        R = P - mid
        Rn = np.linalg.norm(R, axis=-1)
        with np.errstate(divide="ignore", invalid="ignore"):
            B = B + np.cross(dl, R) / Rn[..., None] ** 3
    return MU0 * current / (4.0 * np.pi) * np.nan_to_num(B)


def current_loop(radius, n=400, z0=0.0):
    """A circular current loop in a plane of constant z.

    A closed ring path for :func:`biot_savart`.

    Parameters
    ----------
    radius : float
        Loop radius.
    n : int, optional
        Number of points (default 400).
    z0 : float, optional
        Plane height (default 0.0).

    Returns
    -------
    numpy.ndarray
        The loop path of shape ``(n, 3)``.
    """
    phi = np.linspace(0.0, 2.0 * np.pi, n)
    return np.stack(
        [radius * np.cos(phi), radius * np.sin(phi), np.full_like(phi, z0)], axis=-1
    )


def curl_z(Ex, Ey, x, y):
    """The z-component of the curl of E on a 2-D grid.

    ∂Ey/∂x − ∂Ex/∂y by `numpy.gradient`; nonzero here is the signature of a
    non-conservative induced field.

    Parameters
    ----------
    Ex, Ey : numpy.ndarray
        Field components on the grid.
    x, y : numpy.ndarray
        1-D coordinate arrays.

    Returns
    -------
    numpy.ndarray
        The out-of-plane curl component.
    """
    return np.gradient(Ey, x, axis=0) - np.gradient(Ex, y, axis=1)

## Exercise 1 — Faraday's law: a changing flux

The simplest induction experiment holds a loop still and changes the field through it.
Take a fixed planar loop of area $A=0.01\,\mathrm{m}^2$ in a spatially uniform but
time-varying field $B(t)=B_0\sin(\omega t)$ with $B_0=0.5\,$T and $\omega=2\pi\cdot
50\,\mathrm{s}^{-1}$, the field through the loop ({numref}`fig-ind-faraday-setup`). The
flux is $\Phi_B(t)=A\,B(t)$, and Faraday's law {eq}`eq-faraday` predicts an EMF
$\mathcal{E}=-d\Phi_B/dt=-A B_0\omega\cos(\omega t)$.

**This exercise (worked).**

1. Sample $\Phi_B(t)$ on a uniform time grid over two periods, differentiate
   it numerically with `numpy.gradient` to get $\mathcal{E}=-d\Phi_B/dt$,
   and compare to the closed form $-AB_0\omega\cos(\omega t)$.
2. Validate on the **interior** of the time array (excluding the two end
   samples, where `numpy.gradient` falls back to one-sided differences).

The animation
({numref}`fig-ind-faraday-anim`) is the heart of the exercise: motion is the content
here, so we watch the field through the loop oscillate while the flux and the induced
EMF trace out their curves, a quarter-period apart.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    emf_numeric[interior],
    emf_closed[interior],
    "the induced EMF equals −dΦ/dt (interior, away from the gradient's edge stencils)",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Lenz's law and the sign

The minus sign in {eq}`eq-faraday` is not bookkeeping; it is **Lenz's law**
{eq}`eq-lenz`, and it encodes energy conservation. The induced EMF, and the current it
drives, always opposes the *change* in flux: when $\Phi_B$ is increasing the induced
current fights the increase, when decreasing it props the flux up. A current that did
the opposite would reinforce its own cause and manufacture energy from nothing.

**This exercise (worked).** Reusing the flux of Exercise 1, form $d\Phi_B/dt$
with `numpy.gradient` and confirm that the induced EMF $\mathcal{E}=-d\Phi_B/dt$ carries
the **opposite sign** to $d\Phi_B/dt$ wherever the change is non-negligible, that is,
$\operatorname{sign}\mathcal{E}=-\operatorname{sign}(d\Phi_B/dt)$. We mask out the
instants where $d\Phi_B/dt\approx0$ (the flux extrema), where the sign is numerically
meaningless, using a threshold at 1% of the peak rate.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    bool(opposes),
    "the induced EMF opposes the change in flux (Lenz's law: sign ℰ = −sign dΦ/dt)",
)

## Exercise 3 — ∇×E = −∂B/∂t: the field stops being conservative

Here is the hinge. Confine a uniformly increasing field to a cylinder of radius
$a=0.1\,$m along $\hat{\mathbf z}$, with $dB/dt=1.0\,\mathrm{T/s}$. Symmetry forces the
induced electric field to be **azimuthal**, circling the axis with
$E_\varphi(r)=-\tfrac{r}{2}\,dB/dt$, so in Cartesian components
$\mathbf E=\tfrac12\frac{dB}{dt}\,(y,\,-x,\,0)$ ({numref}`fig-ind-curlE-setup`). This
field winds in closed loops and points at no charge.

**This exercise (worked).**

1. Build $\mathbf E$ on a 3-D grid and take its curl numerically (the
   `curl_z` helper); confirm the $z$-component equals $-dB/dt$ everywhere,
   **nonzero**.
2. Integrate $\oint\mathbf E\cdot d\boldsymbol\ell$ around a circle of
   radius $r_0=0.05\,$m with `numpy.trapezoid` and confirm it equals
   $-d\Phi_B/dt=-\pi r_0^2\,dB/dt\neq0$.

This is precisely the
property [§3.2](potential-energy.ipynb) said would fail once induction entered: the
work around a closed loop is
no longer zero, $\mathbf E$ is no longer conservative, and no single-valued scalar
potential can describe it ({numref}`fig-ind-curlE-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    curlz_interior,
    -dBdt,
    "∇×E = −∂B/∂t (nonzero — the induced E is not conservative)",
    rtol=1e-4,
)
validate.close(
    circulation,
    -flux_rate,
    "∮E·dℓ = −dΦ/dt ≠ 0, breaking the electrostatic path-independence of §3.2",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The AC generator: a rotating loop

Spin a loop in a steady field and the flux changes because the loop's orientation
does: this is how almost all of the world's electricity is made. Take $N=100$ turns of
area $A=0.01\,\mathrm{m}^2$ rotating at $\omega=2\pi\cdot50\,\mathrm{s}^{-1}$ in a
uniform field $B=0.2\,$T ({numref}`fig-ind-generator-setup`). The flux per turn is
$B A\cos(\omega t)$, so $\Phi_B=NBA\cos(\omega t)$ and Faraday's law gives
$\mathcal{E}=-d\Phi_B/dt=NBA\omega\sin(\omega t)$, a sinusoidal AC output with peak
$NBA\omega$.

**This exercise (worked).** Compute $\Phi_B(t)$, differentiate it with
`numpy.gradient` to get $\mathcal{E}$, and confirm against the closed form
$NBA\omega\sin(\omega t)$ on the **interior** of the time array. The flux and EMF come
out a quarter-period (90°) apart, the defining signature of a generator
({numref}`fig-ind-generator-output`). This sinusoidal EMF is exactly the source that
will drive the AC RLC circuits of [§3.11](rlc-circuits.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    emf_gen[interior],
    emf_gen_closed[interior],
    "the rotating loop generates sinusoidal AC, ℰ = NBAω sin(ωt) (interior)",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Motional EMF: the sliding bar

Flux can change with no change in the field at all, simply because the circuit's area
grows. A conducting bar of length $L=0.2\,$m slides at speed $v=5\,$m/s along parallel
rails in a uniform field $B=0.5\,$T perpendicular to the plane of the circuit
({numref}`fig-ind-motional-setup`). Two pictures give the same EMF. The **flux rule**
{eq}`eq-faraday`: the enclosed area grows at $L v$, so $|d\Phi_B/dt|=BLv$. The
**force picture** {eq}`eq-motional`: each carrier feels $q\,\mathbf v\times\mathbf B$,
an effective field of magnitude $vB$ along the bar, which integrated over its length
$L$ gives an EMF $vBL$.

**This exercise (student).**

1. Compute the motional EMF from the flux rule (the rate of area change
   times $B$).
2. Compute it again from the $\mathbf v\times\mathbf B$ force integrated
   along the bar (`numpy.trapezoid`).
3. Confirm they agree and both equal $BLv$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    emf_flux_rule, B_bar * L_bar * v_bar, "the motional EMF is BLv", rtol=1e-6
)
validate.close(
    emf_force,
    emf_flux_rule,
    "the v×B force picture and the flux rule give the same motional EMF",
    rtol=1e-6,
)

## Exercise 6 — Mutual inductance of coaxial loops

When two circuits share flux, the current in one induces an EMF in the other, with
the coupling set by the **mutual inductance** $M$ {eq}`eq-inductance`. Take two coaxial
loops, a large one of radius $r_1=0.1\,$m and a small one of radius $r_2=0.01\,$m on
the same axis, separated by $z=0.05\,$m ({numref}`fig-ind-mutual-setup`). For a small
inner loop the field is nearly uniform across it, so the dipole/on-axis formula gives
$M=\mu_0\pi r_1^2 r_2^2/\bigl(2(r_1^2+z^2)^{3/2}\bigr)$.

**This exercise (worked).** Drive unit current around loop 1 and compute the
field it makes at loop 2 with the Biot–Savart integrator of
[§3.6](magnetostatics.ipynb); the flux
through loop 2 per unit current is $M$. We evaluate $B_z$ at loop 2's centre, multiply
by its area $\pi r_2^2$ (the small-loop approximation), and compare to the closed-form
$M$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    M_from_flux,
    M_formula,
    "the mutual inductance matches the coaxial-loop formula",
    rtol=1e-2,
)

## Exercise 7 — Eddy currents and a magnet falling through a copper pipe

Lenz's law has a famous, almost magical demonstration: drop a magnet down a copper
pipe and it floats down in slow motion. As the magnet falls, the flux through each
ring of the pipe changes, inducing circulating **eddy currents** that, by Lenz's law,
oppose the change, which means they pull back on the magnet ({numref}`fig-ind-eddy-setup`).
The retarding force grows with speed, so the magnet reaches a **terminal velocity** at
which gravity and the magnetic drag balance.

**This exercise (student).** Model the speed $v$ (downward, positive) with a linear
drag from the induced EMF, $m\,\dot v = mg - k\,v$, for a magnet of mass
$m=0.01\,$kg and drag coefficient $k=0.5\,$kg/s under $g=9.81\,\mathrm{m/s^2}$.
Integrate it with `scipy.integrate.solve_ivp` (the `DOP853` high-order
Runge–Kutta) from rest and confirm the speed approaches the terminal value
$v_\infty=mg/k$ ({numref}`fig-ind-eddy-velocity`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    v_fall[-1],
    v_terminal,
    "eddy-current damping gives a terminal velocity (the magnet falls slowly)",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Energy stored in the magnetic field

Building a current against the induced back-EMF takes work, and that work is stored in
the magnetic field as energy $\tfrac12 LI^2$, the magnetic mirror of the electric field
energy of [§3.2](potential-energy.ipynb). For a long solenoid of $n$ turns per metre,
radius $R$, length $\ell$
carrying current $I$, the interior field is $B=\mu_0 nI$ (Ampère,
[§3.6](magnetostatics.ipynb)) and the
self-inductance is $L=\mu_0 n^2(\pi R^2)\ell$ ({numref}`fig-ind-energy-setup`).

**This exercise (student).** For $n=1000\,\mathrm{m^{-1}}$, $R=0.02\,$m,
$\ell=0.4\,$m and $I=2\,$A:

1. Compute the circuit energy $\tfrac12 LI^2$.
2. Independently, integrate the field-energy density $u=B^2/2\mu_0$ over the
   solenoid's interior volume with `numpy.trapezoid`.

The two must agree, the field-energy bookkeeping of [§3.2](potential-energy.ipynb)
carried over to
magnetism.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    energy_field,
    energy_circuit,
    "the field energy ∫(B²/2μ₀) dV equals the circuit energy ½LI²",
    rtol=1e-3,
)

## Exercise 9 — Three of four, and the coupling begins

Step back and collect the field equations. After electrostatics, magnetostatics and
now induction, three of the four are in hand, and one of them has just changed
character:

$$ \nabla\cdot\mathbf E = \frac{\rho}{\varepsilon_0}\ (3.3), \qquad
\nabla\cdot\mathbf B = 0\ (3.6), \qquad
\nabla\times\mathbf E = -\frac{\partial\mathbf B}{\partial t}\ (\text{here}), \qquad
\nabla\times\mathbf B = \mu_0\mathbf J\ (3.6, \text{still static}). $$

The third equation is **new in kind**: for the first time a time derivative couples the
two fields, a changing $\mathbf B$ making an $\mathbf E$. The fourth is still the
static Ampère law, and it is **incomplete**: consistency with charge conservation will
force a symmetric term, a changing $\mathbf E$ making a $\mathbf B$, the displacement
current of [§3.8](maxwell-waves.ipynb). Add that one term and the equations close
into a self-sustaining
system whose solutions propagate: **light**.

One structural consequence is already unavoidable. With $\nabla\times\mathbf E\neq0$
the field can no longer come from a scalar potential alone; it needs the scalar and
vector potentials together, $\mathbf E=-\nabla V-\partial_t\mathbf A$, the full gauge
picture set up in [§3.8](maxwell-waves.ipynb).

**This exercise.** Make the new coupling concrete one last time: reusing the
`curl_z` result of Exercise 3, confirm *quantitatively* that the induced
field's curl equals $-\partial_t\mathbf B$ — the one dynamic term now
switched on — while the displacement-current twin $\partial_t\mathbf E$ is
still identically zero in this notebook. We are one term, and one notebook,
from light.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    curlE_residual,
    0.0,
    "the one dynamic term switched on, measured: ∇×E = −∂B/∂t; its ∂E/∂t twin awaits §3.8",
    atol=1e-4,
)

## Notebook summary

- **Faraday's law** {eq}`eq-faraday`: a changing flux drives $\mathcal{E}=-d\Phi_B/dt$;
  we differentiated a sampled flux with `numpy.gradient` and matched the closed form on
  the interior (Exercise 1, the warranted animation).
- **Lenz's law** {eq}`eq-lenz`: the induced EMF opposes the change in flux, energy
  conservation made into a sign (Exercise 2).
- **The hinge** {eq}`eq-curlE`: for a region of changing $\mathbf B$ the induced
  $\mathbf E$ circulates, $\nabla\times\mathbf E=-\partial_t\mathbf B\neq0$ and
  $\oint\mathbf E\cdot d\boldsymbol\ell\neq0$, breaking the path-independence of
  [§3.2](potential-energy.ipynb) (Exercise 3).
- **The AC generator** (Exercise 4): a rotating loop gives $\mathcal{E}=NBA\omega
  \sin\omega t$, sinusoidal AC from first principles, the source for
  [§3.11](rlc-circuits.ipynb).
- **Motional EMF** {eq}`eq-motional`: the sliding bar gives $BLv$ by the flux rule and
  the $\mathbf v\times\mathbf B$ force alike (Exercise 5).
- **Inductance and energy** {eq}`eq-inductance`: the mutual inductance of coaxial loops
  from a Biot–Savart flux (Exercise 6), and the magnetic field energy $\tfrac12 LI^2 =
  \int B^2/2\mu_0\,dV$ (Exercise 8); eddy-current braking and a terminal velocity for a
  magnet in a pipe (Exercise 7).
- **Three of four field equations** assembled, with $\nabla\times\mathbf E$ now
  dynamic and the displacement current of [§3.8](maxwell-waves.ipynb) the last piece
  before light (Exercise 9).

## Outlook

- **The betatron.** Induction is not just a generator trick: the circulating induced
  $\mathbf E$ of Exercise 3 can *accelerate particles*. A betatron ramps a magnetic
  field to drive electrons around a ring with no electrodes at all, the induced field
  doing the work.
- **The transformer.** Mutual inductance (Exercise 6) at industrial scale: two coils
  sharing a flux step voltages up or down, the device that makes the AC power grid
  practical.
- **AC circuits and impedance ([§3.11](rlc-circuits.ipynb)).** The sinusoidal EMF of
  the generator (Exercise 4)
  is exactly the source that drives the RLC resonator; inductors there contribute a
  reactance $\omega L$, the circuit face of the $\tfrac12 LI^2$ energy met here.
- **The potential formulation and the Lorenz gauge ([§3.8](maxwell-waves.ipynb)).** Once
  $\nabla\times\mathbf E\neq0$ we need $\mathbf E=-\nabla V-\partial_t\mathbf A$; the
  gauge arc begun in [§3.6](magnetostatics.ipynb) continues there and closes with
  electromagnetic waves.
- **Superconductors and flux quantization (Vol VI).** A persistent current flows
  forever with no driving EMF, and the trapped flux comes in discrete quanta $h/2e$, a
  macroscopic quantum effect that this classical picture can only point toward.